In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_solutions.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/sample_submission.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json


In [2]:
# ==============================================================================
# DR. T PATIENT HEART COMPANION & CLINICAL NLP - KAGGLE OFFLINE EXECUTION
# ==============================================================================
# Target Notebook: https://www.kaggle.com/code/zenieverse/notebookc05162cafa/edit
# Author: Zenieverse / Dr. T Clinical R&D Suite
# Environment: Kaggle Notebooks (CPU / GPU T4 / P100 / TPU)
# Execution Mode: 100% Offline (Zero Internet Connection Required)
# ==============================================================================

# ------------------------------------------------------------------------------
# CELL 1: ENVIRONMENT SETUP & DEPENDENCY IMPORTS
# ------------------------------------------------------------------------------
import os
import re
import json
import time
import math
import random
import datetime
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 75)
print("DR. T CLINICAL NLP & PATIENT HEART COMPANION - OFFLINE KAGGLE PIPELINE")
print("Target Notebook: https://www.kaggle.com/code/zenieverse/notebookc05162cafa/edit")
print("=" * 75)

# Check PyTorch & CUDA Availability for Offline Hardware Acceleration
try:
    import torch
    print(f"[+] PyTorch Version: {torch.__version__}")
    print(f"[+] CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"[+] Active GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    print("[!] PyTorch not detected. Running in standard CPU/NumPy fallback mode.")
    device = "cpu"


# ------------------------------------------------------------------------------
# CELL 2: HIPAA & GDPR PII/PHI ANONYMIZATION ENGINE
# ------------------------------------------------------------------------------
class HIPAAAnonymizer:
    """
    De-identifies Patient Health Information (PHI) and Personally Identifiable
    Information (PII) for secure R&D clinical repository archiving offline.
    """
    def __init__(self):
        self.patterns = {
            'email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
            'phone': r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}',
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'mrn': r'\bMRN[-:\s]?\d{6,10}\b|\bID[-:\s]?\d{6,10}\b',
            'date': r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}\b',
            'doctor': r'\bDr\.\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?\b',
            'location': r'\b(?:Hospital|Clinic|Medical Center|Ward|ICU|Room)\s+\d+[A-Za-z]?\b'
        }

    def anonymize(self, text: str, patient_id_seq: int = 1001):
        sanitized = text
        sanitized = re.sub(self.patterns['email'], '[ANONYMIZED_EMAIL]', sanitized)
        sanitized = re.sub(self.patterns['phone'], '[ANONYMIZED_PHONE]', sanitized)
        sanitized = re.sub(self.patterns['ssn'], '[ANONYMIZED_SSN]', sanitized)
        sanitized = re.sub(self.patterns['mrn'], '[ANONYMIZED_MRN]', sanitized)
        sanitized = re.sub(self.patterns['date'], '[ANONYMIZED_DATE]', sanitized)
        sanitized = re.sub(self.patterns['doctor'], '[ANONYMIZED_PHYSICIAN]', sanitized)
        sanitized = re.sub(self.patterns['location'], '[ANONYMIZED_FACILITY]', sanitized)
        
        alias = f"PATIENT_ANON_{patient_id_seq}"
        return sanitized, alias

anonymizer = HIPAAAnonymizer()
print("[+] HIPAA/GDPR Anonymizer initialized successfully.")


# ------------------------------------------------------------------------------
# CELL 3: CLINICAL EMOTION & SYMPTOM EXTRACTOR (NLP PIPELINE)
# ------------------------------------------------------------------------------
class ClinicalNLPExtractor:
    """
    Offline Rule & Lexicon NLP Extractor for Patient Heart Narratives.
    Detects Emotional States, Physical Symptoms, Treatment Phase, and R&D Insights.
    """
    EMOTION_DICTIONARY = {
        'Anxiety & Fear': ['anxious', 'terrified', 'scared', 'fear', 'panic', 'dread', 'worried', 'nervous', 'scared of needles', 'afraid'],
        'Exhaustion & Fatigue': ['exhausted', 'tired', 'drained', 'fatigue', 'weakness', 'no energy', 'burnout', 'heavy'],
        'Physical Pain & Discomfort': ['pain', 'hurt', 'aching', 'cramps', 'throbbing', 'sore', 'burning', 'sharp pain', 'stabbing'],
        'Emotional Isolation & Sadness': ['lonely', 'isolated', 'crying', 'sad', 'depressed', 'hopeless', 'overwhelmed', 'no one understands'],
        'Hope & Resilience': ['hopeful', 'trying', 'determined', 'better today', 'thankful', 'fighting', 'grateful', 'optimistic'],
        'Nausea & Side Effects': ['nausea', 'vomiting', 'sick to stomach', 'queasy', 'dizzy', 'loss of appetite', 'metallic taste']
    }

    SYMPTOM_DICTIONARY = {
        'Needle Phobia / Vein Fragility': ['needle', 'vein', 'iv', 'phlebotomy', 'puncture', 'bruising'],
        'Chemo / Infusion Nausea': ['nausea', 'infusion', 'chemo', 'drip', 'vomit', 'appetite'],
        'Insomnia & Sleep Disturbance': ['insomnia', 'cannot sleep', 'awake all night', 'restless'],
        'Pagophagia / Pica': ['ice', 'chewing ice', 'ice craving', 'crunching ice'],
        'Cardiovascular Palpitations': ['heart racing', 'palpitations', 'fluttering', 'chest tightness', 'short of breath'],
        'Neuropathy / Tingling': ['tingling', 'numbness', 'pins and needles', 'fingertips']
    }

    def analyze(self, text: str):
        text_lower = text.lower()
        
        # 1. Detect Emotions
        detected_emotions = []
        for emotion, keywords in self.EMOTION_DICTIONARY.items():
            if any(kw in text_lower for kw in keywords):
                detected_emotions.append(emotion)
        if not detected_emotions:
            detected_emotions = ['General Vulnerability / Emotional Stress']

        # 2. Detect Symptoms
        detected_symptoms = []
        for symptom, keywords in self.SYMPTOM_DICTIONARY.items():
            if any(kw in text_lower for kw in keywords):
                detected_symptoms.append(symptom)
        if not detected_symptoms:
            detected_symptoms = ['General Treatment Fatigue / Malaise']

        # 3. Formulate R&D Recommendations
        rnd_takeaways = []
        if 'Needle Phobia / Vein Fragility' in detected_symptoms:
            rnd_takeaways.append("R&D Medical Device Rec: Implement vein-visualization IR scanner & micro-needle infusion ports to lower phlebotomy trauma.")
        if 'Chemo / Infusion Nausea' in detected_symptoms:
            rnd_takeaways.append("R&D Pharma Rec: Formulate fast-acting sublingual anti-emetic strips prior to infusion cycles.")
        if 'Pagophagia / Pica' in detected_symptoms:
            rnd_takeaways.append("R&D Diagnostics Rec: Pagophagia is a strong biomarker for severe iron deficiency anemia; trigger automated ferritin lab panels.")
        if not rnd_takeaways:
            rnd_takeaways.append("R&D Care Quality Rec: Enhance patient-centered empathetic check-ins and provide personalized recovery schedules.")

        # 4. Phase Categorization
        phase = "Pre-Treatment / Prep"
        if any(k in text_lower for k in ['during', 'chemo', 'infusion', 'active', 'daily dose', 'side effect']):
            phase = "Mid-Treatment / Active Care"
        elif any(k in text_lower for k in ['after', 'finished', 'post', 'remission', 'recovery', 'weeks later']):
            phase = "Post-Treatment / Recovery"

        return {
            'emotions': detected_emotions,
            'symptoms': detected_symptoms,
            'rnd_recommendations': " | ".join(rnd_takeaways),
            'phase': phase
        }

extractor = ClinicalNLPExtractor()
print("[+] Clinical NLP Extractor loaded successfully.")


# ------------------------------------------------------------------------------
# CELL 4: DR. T EMPATHETIC HEART-TO-HEART RESPONSE GENERATOR (OFFLINE AI)
# ------------------------------------------------------------------------------
class DrTEmpatheticGenerator:
    """
    Offline Empathetic Response Generator simulating Dr. T's compassionate
    bedside manner and verified medical grounding.
    """
    def generate_response(self, text: str, analysis: dict, phase: str):
        emotions_str = ", ".join(analysis['emotions'])
        symptoms_str = ", ".join(analysis['symptoms'])

        response = (
            f"Dear heart, I hear you so deeply. Pouring your feelings out about {symptoms_str.lower()} "
            f"takes immense bravery. It is completely natural to experience {emotions_str.lower()} "
            f"during the {phase} stage.\n\n"
            f"Please remember that your feelings are valid and your body is fighting hard for you every second. "
            f"Here are 3 supportive clinical care steps you can take right now:\n"
            f"1. Rest and hydrate with warm fluids or electrolyte-rich solutions.\n"
            f"2. Practice gentle 4-7-8 diaphragmatic breathing to calm your nervous system.\n"
            f"3. Note down these exact symptoms so we can review them together with your care team.\n\n"
            f"I am standing right beside you through this journey. You are never alone."
        )
        return response

generator = DrTEmpatheticGenerator()
print("[+] Dr. T Empathetic Generator active.")


# ------------------------------------------------------------------------------
# CELL 5: BATCH PROCESSING DATASET & KAGGLE EXPORT PIPELINE
# ------------------------------------------------------------------------------
sample_patient_narratives = [
    "I'm so scared of my upcoming chemotherapy infusion tomorrow. The needles always hurt and my veins collapse. I can't stop crying. Contact me at patient@example.com or Dr. Smith.",
    "Day 5 after my procedure and the nausea is unbearable. I've been craving ice constantly and crunching on ice cubes all day.",
    "Finished my 6th cycle last week! I feel so fatigued and brain foggy, but I'm trying to remain hopeful for my recovery scan."
]

processed_records = []

print("\nProcessing Patient Narratives Offline...")
for idx, raw_story in enumerate(sample_patient_narratives, start=101):
    clean_text, alias = anonymizer.anonymize(raw_story, patient_id_seq=idx)
    nlp_res = extractor.analyze(clean_text)
    response_text = generator.generate_response(clean_text, nlp_res, nlp_res['phase'])

    record = {
        'caseId': f"RND-KAG-{idx}",
        'patientAlias': alias,
        'treatmentPhase': nlp_res['phase'],
        'patientStory': clean_text,
        'emotionsDetected': nlp_res['emotions'],
        'symptomsDetected': nlp_res['symptoms'],
        'rndResearchInsights': nlp_res['rnd_recommendations'],
        'drTResponse': response_text,
        'anonymizedSummary': f"Offline Kaggle case analyzing {', '.join(nlp_res['emotions'])} and {', '.join(nlp_res['symptoms'])}.",
        'createdAt': datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    processed_records.append(record)

# Convert to pandas DataFrame
df = pd.DataFrame(processed_records)
print("\n" + "=" * 75)
print("PROCESSED R&D CLINICAL CASES (DATAFRAME PREVIEW):")
print("=" * 75)
print(df[['caseId', 'patientAlias', 'treatmentPhase', 'rndResearchInsights']].to_string(index=False))

# Export to CSV & JSON
csv_path = "drt_rnd_cases_kaggle_export.csv"
json_path = "drt_rnd_cases_kaggle_export.json"

df.to_csv(csv_path, index=False)
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(processed_records, f, indent=2)

print(f"\n[✓] Successfully exported {len(processed_records)} cases to '{csv_path}' and '{json_path}'!")
print("[✓] Ready for offline Kaggle competition submission or direct import into Dr. T Health Repository app!")

DR. T CLINICAL NLP & PATIENT HEART COMPANION - OFFLINE KAGGLE PIPELINE
Target Notebook: https://www.kaggle.com/code/zenieverse/notebookc05162cafa/edit
[+] PyTorch Version: 2.10.0+cpu
[+] CUDA Available: False
[+] HIPAA/GDPR Anonymizer initialized successfully.
[+] Clinical NLP Extractor loaded successfully.
[+] Dr. T Empathetic Generator active.

Processing Patient Narratives Offline...

PROCESSED R&D CLINICAL CASES (DATAFRAME PREVIEW):
     caseId     patientAlias              treatmentPhase                                                                                                                                                                                                             rndResearchInsights
RND-KAG-101 PATIENT_ANON_101 Mid-Treatment / Active Care       R&D Medical Device Rec: Implement vein-visualization IR scanner & micro-needle infusion ports to lower phlebotomy trauma. | R&D Pharma Rec: Formulate fast-acting sublingual anti-emetic strips prior to infusion cycle